# Brand Ads OCR — **CPU build** (DeepSeek‑OCR wired)

A single notebook that:
1) Seeds/loads a manifest of Apple/Samsung/Huawei/Yamaha ad pages
2) Fetches the hero creative image from each page (robust meta/UA)
3) Runs **DeepSeek‑OCR** on **CPU** via `transformers` with `trust_remote_code=True` and `model.infer(...)`
4) Produces `data/outputs/predictions.csv` with `brand_pred`, `entity`, and full OCR text

This notebook is tuned for CPU (no Flash-Attn, smaller image sizes).

## 0) Install (CPU)
Run these in your environment if needed. Keep Python ≤ 3.12.

In [1]:
# %%bash
# pip install --upgrade pip
# pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0   # CPU wheels
# pip install transformers==4.46.3 tokenizers==0.20.3 einops addict easydict
# pip install requests beautifulsoup4 pillow pandas
print('Install commands (commented) ready for CPU setup.')

Install commands (commented) ready for CPU setup.


## 1) Paths & helpers

In [2]:
import os, re, io, csv, time, json, glob, pathlib
from typing import Optional
import pandas as pd
import requests
from bs4 import BeautifulSoup
from PIL import Image
from urllib.parse import urljoin

ROOT = pathlib.Path('.')
DATA = ROOT / 'data'
IMAGES = DATA / 'images'
OUT = DATA / 'outputs'
OCR_JSON = OUT / 'ocr_json'
for p in [DATA, IMAGES, OUT, OCR_JSON]:
    p.mkdir(parents=True, exist_ok=True)

def slug(s: str) -> str:
    return re.sub(r'[^a-zA-Z0-9]+', '_', s).strip('_')

# Robust session with UA
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.adsoftheworld.com/",
})

def _find_image_meta(soup: BeautifulSoup) -> Optional[str]:
    metas = [
        ("meta", {"property":"og:image"}),
        ("meta", {"property":"og:image:secure_url"}),
        ("meta", {"name":"twitter:image"}),
        ("meta", {"name":"twitter:image:src"}),
    ]
    for tag, attrs in metas:
        el = soup.find(tag, attrs=attrs)
        if el and el.get("content"):
            return el["content"]
    return None

def _find_first_large_img(soup: BeautifulSoup, base_url: str) -> Optional[str]:
    best, best_area = None, 0
    for img in soup.find_all("img"):
        src = img.get("src") or img.get("data-src")
        if not src:
            continue
        src = urljoin(base_url, src)
        w = img.get("width"); h = img.get("height")
        try:
            area = int(w)*int(h) if w and h else 0
        except:
            area = 0
        score = area
        if re.search(r"(1200|1080|1024|800)", src):
            score += 2000
        if score > best_area:
            best_area, best = score, src
    return best

def og_image(url: str) -> Optional[str]:
    try:
        r = SESSION.get(url, timeout=30, allow_redirects=True)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        img = _find_image_meta(soup)
        if img:
            return img
        time.sleep(0.5)
        r2 = SESSION.get(url, timeout=30, allow_redirects=True)
        soup2 = BeautifulSoup(r2.text, "html.parser")
        img = _find_image_meta(soup2)
        if img:
            return img
        return _find_first_large_img(soup2, url)
    except Exception as e:
        print("OG image error:", url, e)
        return None

## 2) Manifest (20 ads)

In [3]:
manifest_path = DATA / 'manifest.csv'
if not manifest_path.exists():
    rows = [
        # Apple (5)
        {"brand":"Apple","title":"Come Rain or Come Shine","medium":"Film","year":"","region":"","page_url":"https://www.adsoftheworld.com/campaigns/come-rain-or-come-shine"},
        {"brand":"Apple","title":"Shot on iPhone","medium":"OOH/Print/Film","year":"2019","region":"US","page_url":"https://www.adsoftheworld.com/campaigns/shot-on-iphone"},
        {"brand":"Apple","title":"Your next computer is not a computer","medium":"Film/Digital","year":"2021","region":"US","page_url":"https://www.adsoftheworld.com/campaigns/your-next-computer-is-not-a-computer"},
        {"brand":"Apple","title":"Mac goes to College","medium":"Film/Digital","year":"2025","region":"US","page_url":"https://www.adsoftheworld.com/campaigns/mac-goes-to-college"},
        {"brand":"Apple","title":"Dear Apple","medium":"Film/Digital","year":"2022","region":"US","page_url":"https://www.adsoftheworld.com/campaigns/dear-apple"},
        # Samsung (5)
        {"brand":"Samsung","title":"The Real Upgrade","medium":"Film/Digital","year":"","region":"","page_url":"https://www.adsoftheworld.com/campaigns/the-real-upgrade"},
        {"brand":"Samsung","title":"No reflections","medium":"Print","year":"2020","region":"Germany","page_url":"https://www.adsoftheworld.com/campaigns/no-reflections"},
        {"brand":"Samsung","title":"Experience the wonder with the Galaxy","medium":"Film/Digital","year":"2021","region":"Mongolia","page_url":"https://www.adsoftheworld.com/campaigns/experience-the-wonder-with-the-galaxy"},
        {"brand":"Samsung","title":"Big News","medium":"Film/Digital","year":"2023","region":"Sweden","page_url":"https://www.adsoftheworld.com/campaigns/big-news"},
        {"brand":"Samsung","title":"For life with you","medium":"Film/Digital","year":"2025","region":"Brazil","page_url":"https://www.adsoftheworld.com/campaigns/for-the-life-with-you"},
        # Huawei (5)
        {"brand":"Huawei","title":"World, Unfolded.","medium":"Film","year":"2025","region":"","page_url":"https://www.adsoftheworld.com/campaigns/huawei-world-unfolded"},
        {"brand":"Huawei","title":"Giant Smart","medium":"Print/OOH","year":"2025","region":"AU/BR/FR","page_url":"https://www.adsoftheworld.com/campaigns/huawei-giant-smart"},
        {"brand":"Huawei","title":"Welcome to the Huawei AppGallery","medium":"Digital/Film","year":"2020","region":"Israel","page_url":"https://www.adsoftheworld.com/campaigns/welcome-to-the-huawei-appgallery"},
        {"brand":"Huawei","title":"The Unofficial Smartphone","medium":"Film/Digital","year":"2018","region":"Mexico","page_url":"https://www.adsoftheworld.com/campaigns/the-unofficial-smartphone"},
        {"brand":"Huawei","title":"Inspiration In Bloom","medium":"OOH/Print","year":"2025","region":"BR/FR/UK","page_url":"https://www.adsoftheworld.com/campaigns/huawei-inspiration-in-bloom"},
        # Yamaha (5)
        {"brand":"Yamaha","title":"Reveal","medium":"Print/OOH","year":"2014","region":"Italy","page_url":"https://www.adsoftheworld.com/campaigns/reveal"},
        {"brand":"Yamaha","title":"Off-ROAD MILKSHAKE","medium":"Activation/Film","year":"2025","region":"","page_url":"https://www.adsoftheworld.com/campaigns/off-road-milkshake"},
        {"brand":"Yamaha","title":"Neurons","medium":"Print","year":"2017","region":"Italy","page_url":"https://www.adsoftheworld.com/campaigns/neurons"},
        {"brand":"Yamaha","title":"Balls","medium":"Print","year":"2016","region":"Israel","page_url":"https://www.adsoftheworld.com/campaigns/balls-74ffa437-ae4d-4cd2-b8e7-b8d6596db2e4"},
        {"brand":"Yamaha","title":"Paper City","medium":"Print","year":"2011","region":"Italy","page_url":"https://www.adsoftheworld.com/campaigns/paper-city-31cdab6b-2bea-4394-be46-0dd3ffcba459"}
    ]
    pd.DataFrame(rows).to_csv(manifest_path, index=False)
manifest_path

PosixPath('data/manifest.csv')

## 3) Fetch images

In [4]:
df = pd.read_csv(manifest_path)
downloaded = []
for _, row in df.iterrows():
    page = row['page_url']
    name = f"{row['brand']}_{slug(row['title'])}.jpg"
    path = IMAGES / name
    if path.exists():
        downloaded.append(str(path)); continue
    img_url = og_image(page)
    if not img_url:
        print('No image', page)
        continue
    try:
        resp = SESSION.get(img_url, timeout=30)
        resp.raise_for_status()
        with open(path, 'wb') as f:
            f.write(resp.content)
        downloaded.append(str(path))
        time.sleep(0.4)
    except Exception as e:
        print('Download error', img_url, e)
len(downloaded), downloaded[:3]

(20,
 ['data/images/Apple_Come_Rain_or_Come_Shine.jpg',
  'data/images/Apple_Shot_on_iPhone.jpg',
  'data/images/Apple_Your_next_computer_is_not_a_computer.jpg'])

## 4) DeepSeek‑OCR on CPU

In [5]:
import torch
from transformers import AutoModel, AutoTokenizer

device = 'cpu'
model_name = 'deepseek-ai/DeepSeek-OCR'

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    use_safetensors=True,
    _attn_implementation='sdpa',
    torch_dtype=torch.float32,
    device_map='cpu'
).eval()

OCR_PROMPT = "<image>\nFree OCR."

def deepseek_ocr_image(image_path: str) -> str:
    res = model.infer(
        tokenizer,
        prompt=OCR_PROMPT,
        image_file=image_path,
        output_path=str(OUT),
        base_size=768,
        image_size=512,
        crop_mode=False,
        save_results=False,
        test_compress=False,
    )
    if isinstance(res, dict):
        return res.get('text', '') or res.get('output', '') or ''
    return str(res) if res is not None else ''

print('DeepSeek‑OCR loaded on CPU (SDPA).')

ImportError: cannot import name 'LlamaFlashAttention2' from 'transformers.models.llama.modeling_llama' (/Users/jjburrell/Econometrics/Econometrics/.venv/lib/python3.13/site-packages/transformers/models/llama/modeling_llama.py)

## 5) Brand/entity rules

In [6]:
import re
CANON = ["apple","samsung","huawei","yamaha"]
ALIASES = {
    "apple":   [r"\biphone\b", r"\bipad\b", r"\bmac\b", r"\bapple watch\b", r"\bairpods\b", r"\bapple\b"],
    "samsung": [r"\bgalaxy\b", r"\bsamsung\b", r"\bz flip\b", r"\bs24\b", r"\bnote\b"],
    "huawei":  [r"\bhuawei\b", r"\bmate\b", r"\bpura\b", r"\bnova\b"],
    "yamaha":  [r"\byamaha\b", r"\byzf\b", r"\bfz\b", r"\br1\b", r"\bmt-0?\d\b", r"\brevstar\b", r"\bclavinova\b"],
}
BLACKLIST = set(CANON + ["iphone","ipad","mac","airpods","watch","galaxy","mate","pura","nova","yzf","fz","r1"]) 

def detect_brand(ocr_text: str) -> str:
    t = ocr_text.lower()
    for brand, pats in ALIASES.items():
        for p in pats:
            if re.search(p, t):
                return brand.capitalize()
    return "Unknown"

def short_entity(ocr_text: str, max_words: int = 6) -> str:
    words = re.findall(r"[a-zA-Z]+", ocr_text.lower())
    keep = [w for w in words if w not in BLACKLIST and 3 <= len(w) <= 12]
    return " ".join(keep[:max_words]) or "generic scene"

## 6) OCR all images → JSON cache

In [7]:
records = []
for img_path in sorted(IMAGES.glob('*.jpg')):
    base = img_path.stem
    out = OCR_JSON / f"{base}.json"
    if out.exists():
        rec = json.load(open(out))
        records.append(rec)
        continue
    text = deepseek_ocr_image(str(img_path))
    rec = {"image": base, "text": text}
    json.dump(rec, open(out, 'w'))
    records.append(rec)
len(records), records[:2] if records else []

NameError: name 'deepseek_ocr_image' is not defined

## 7) Predictions table

In [8]:
pred_rows = []
for rec in records:
    text = rec.get('text','')
    pred_rows.append({
        'image': rec['image'],
        'brand_pred': detect_brand(text),
        'entity': short_entity(text),
        'ocr_text': text
    })
pred_df = pd.DataFrame(pred_rows)
pred_csv = OUT / 'predictions.csv'
pred_df.to_csv(pred_csv, index=False)
pred_df.head(10)

""


## 8) Sanity check

In [9]:
samples = sorted((IMAGES).glob('*.jpg'))
print('Found', len(samples), 'images')
if samples:
    print('Testing OCR on:', samples[0].name)
    print(deepseek_ocr_image(str(samples[0]))[:400])
else:
    print('No images were downloaded. Re-run cell 3; if some pages still fail, open one URL to verify meta tags.')

Found 20 images
Testing OCR on: Apple_Come_Rain_or_Come_Shine.jpg


NameError: name 'deepseek_ocr_image' is not defined